In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("saas_usage.db")

In [3]:
pd.read_sql("""
WITH usage_summary AS (
    SELECT
        u.account_id,
        SUM(CASE
            WHEN u.event_date >= r.recent_30_day_start
            AND u.event_date <= r.reporting_date
            THEN 1 ELSE 0
        END) AS last_30_day_usage,
        SUM(CASE
            WHEN u.event_date >= r.recent_60_day_start
            AND u.event_date < r.recent_30_day_start
            THEN 1 ELSE 0
        END) AS previous_30_day_usage
    FROM usage_events AS u
    CROSS JOIN reporting_context AS r
    GROUP BY u.account_id
),
ticket_summary AS (
    SELECT
        s.account_id,
        COUNT(s.ticket_id) AS recent_high_priority_tickets
    FROM support_tickets AS s
    CROSS JOIN reporting_context AS r
    WHERE LOWER(s.priority) = 'high'
        AND s.created_date >= r.recent_30_day_start
        AND s.created_date <= r.reporting_date
    GROUP BY s.account_id
)
SELECT
    a.account_id,
    a.account_name,
    u.last_30_day_usage,
    u.previous_30_day_usage,
    ROUND(
        100.0 * (u.previous_30_day_usage - u.last_30_day_usage)
        / u.previous_30_day_usage, 2
    ) AS percent_drop,
    t.recent_high_priority_tickets,
    'Significant usage drop and recent high-priority support tickets'
        AS risk_reason
FROM accounts AS a
JOIN usage_summary AS u
    ON a.account_id = u.account_id
JOIN ticket_summary AS t
    ON a.account_id = t.account_id
WHERE a.account_status = 'active'
    AND u.previous_30_day_usage > 0
    AND u.last_30_day_usage <= u.previous_30_day_usage * 0.5
    AND t.recent_high_priority_tickets >= 1
ORDER BY percent_drop DESC, recent_high_priority_tickets DESC;
""", conn)

,account_id,account_name,last_30_day_usage,previous_30_day_usage,percent_drop,recent_high_priority_tickets,risk_reason
0,3,Account_3,25,75,66.67,1,Significant usage drop and recent high-priorit...
1,7,Account_7,20,60,66.67,1,Significant usage drop and recent high-priorit...
2,1,Account_1,30,80,62.50,2,Significant usage drop and recent high-priorit...
3,9,Account_9,20,50,60.00,1,Significant usage drop and recent high-priorit...
4,11,Account_11,20,40,50.00,1,Significant usage drop and recent high-priorit...


In [4]:
pd.read_sql("""
WITH usage_summary AS (
    SELECT
        u.account_id,
        SUM(CASE
            WHEN u.event_date >= r.recent_30_day_start
            AND u.event_date <= r.reporting_date
            THEN 1 ELSE 0
        END) AS last_30_day_usage,
        SUM(CASE
            WHEN u.event_date >= r.recent_60_day_start
            AND u.event_date < r.recent_30_day_start
            THEN 1 ELSE 0
        END) AS previous_30_day_usage
    FROM usage_events AS u
    CROSS JOIN reporting_context AS r
    GROUP BY u.account_id
),
support_summary AS (
    SELECT
        s.account_id,
        COUNT(s.ticket_id) AS recent_high_priority_tickets
    FROM support_tickets AS s
    CROSS JOIN reporting_context AS r
    WHERE s.created_date >= r.recent_30_day_start
        AND s.created_date <= r.reporting_date
        AND s.priority = 'high'
    GROUP BY s.account_id
)
SELECT
    a.account_id,
    a.account_name,
    u.last_30_day_usage,
    u.previous_30_day_usage,
    ROUND(
        100.0 * (u.previous_30_day_usage - u.last_30_day_usage)
        / u.previous_30_day_usage, 2
    ) AS percent_drop,
    s.recent_high_priority_tickets,
    'Significant usage drop and recent high-priority support tickets'
        AS risk_reason
FROM accounts AS a
JOIN usage_summary AS u
    ON a.account_id = u.account_id
JOIN support_summary AS s
    ON a.account_id = s.account_id
WHERE a.account_status = 'active'
    AND u.previous_30_day_usage > 0
    AND u.last_30_day_usage <= u.previous_30_day_usage * 0.5
    AND s.recent_high_priority_tickets > 0
ORDER BY percent_drop DESC, recent_high_priority_tickets DESC;
""", conn)

,account_id,account_name,last_30_day_usage,previous_30_day_usage,percent_drop,recent_high_priority_tickets,risk_reason
0,3,Account_3,25,75,66.67,1,Significant usage drop and recent high-priorit...
1,7,Account_7,20,60,66.67,1,Significant usage drop and recent high-priorit...
2,1,Account_1,30,80,62.50,2,Significant usage drop and recent high-priorit...
3,9,Account_9,20,50,60.00,1,Significant usage drop and recent high-priorit...
4,11,Account_11,20,40,50.00,1,Significant usage drop and recent high-priorit...
